# L12：深度学习实战项目


# L12：深度学习实战项目

## 学习目标

1. 掌握从问题分析到模型设计的完整项目流程
2. 能够构建端到端的图像分类pipeline
3. 理解数据增强、迁移学习在实战中的应用
4. 掌握模型评估与调优的实用技巧
5. 具备独立完成完整深度学习项目的能力

## 核心知识点

### 1. 项目流程概览


In [ ]:
问题定义 -> 数据收集 -> 数据预处理 -> 模型设计 -> 训练 -> 评估 -> 部署


### 2. 问题定义与数据收集

- **明确任务类型**：分类、检测、分割、生成、序列处理等
- **评估指标选择**：
  - 分类：准确率、精确率、召回率、F1、AUC
  - 目标检测：mAP、IoU
  - 分割：IoU、Dice系数
- **数据来源**：公开数据集、网络爬取、传感器采集、合成数据

### 3. 数据预处理核心技巧

#### 图像数据

- **归一化**：将像素值从[0,255]映射到[0,1]或[-1,1]
- **数据增强**：
  - 几何变换：随机裁剪、水平翻转、旋转、缩放
  - 色彩变换：亮度、对比度、饱和度、色调
  - 噪声注入：高斯噪声、椒盐噪声
  - MixUp、CutMix：高级增强技术

#### 文本数据

- **分词（Tokenization）**：WordPiece、BPE、SentencePiece
- **词嵌入**：预训练词向量、GloVe、Word2Vec
- **序列填充**：padding到固定长度

### 4. 模型选择策略

| 任务类型 | 推荐模型 | 说明 |
|---------|---------|------|
| 图像分类 | ResNet、EfficientNet、ViT | 迁移学习的最佳起点 |
| 目标检测 | YOLO、Faster R-CNN、SSD | 实时性 vs 精度权衡 |
| 语义分割 | U-Net、DeepLabV3、Mask R-CNN | 医学影像常用U-Net |
| 文本分类 | BERT、RoBERTa、DistilBERT | Transformer架构 |
| 序列生成 | LSTM、GPT、T5 | 自回归或Seq2Seq |

### 5. 训练技巧

#### 学习率策略

- **学习率预热（Warmup）**：开始时使用较小学习率，逐渐增加到正常值
- **余弦退火（Cosine Annealing）**：学习率按余弦曲线周期性变化
- **ReduceLROnPlateau**：监控指标，当停滞时降低学习率
- **OneCycleLR**：先升后降的单周期策略

#### 正则化

- **Dropout**：训练时随机置零部分神经元
- **L2正则化（权重衰减）**：在损失函数中添加权重范数
- **早停（Early Stopping）**：验证集性能不再提升时停止训练
- **Batch Normalization**：提供轻微的正则化效果

#### 批处理策略

- **Batch Size**：大batch加速训练但泛化性能可能下降
- **Gradient Accumulation**：模拟大batch效果

### 6. 模型评估与调优

- **混淆矩阵**：分析分类错误的模式
- **ROC曲线与AUC**：评估分类器的概率校准
- **PR曲线**：处理类别不平衡
- **迁移学习微调策略**：
  - 冻结底层，逐步解冻
  - 使用较小学习率
  - 渐进式图像尺寸增大

## 代码示例

### 实战项目：花卉分类器（完整Pipeline）


In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

# ============================================================
# 配置
# ============================================================

class Config:
    # 数据路径
    data_dir = "./flower_data"
    train_dir = os.path.join(data_dir, "train")
    val_dir = os.path.join(data_dir, "val")
    test_dir = os.path.join(data_dir, "test")

    # 训练参数
    num_epochs = 30
    batch_size = 32
    learning_rate = 0.001
    num_workers = 4

    # 模型参数
    num_classes = 5
    image_size = 224
    pretrained = True

    # 设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 类别名称
    class_names = ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


# ============================================================
# 数据集定义
# ============================================================

class FlowerDataset(Dataset):
    """花卉数据集"""

    def __init__(self, root_dir, transform=None, mode='train'):
        self.root_dir = root_dir
        self.transform = transform
        self.mode = mode
        self.samples = []
        self.labels = []

        # 加载数据
        for label_idx, class_name in enumerate(Config.class_names):
            class_dir = os.path.join(root_dir, class_name)
            if not os.path.exists(class_dir):
                continue
            for img_name in os.listdir(class_dir):
                if img_name.endswith(('.jpg', '.jpeg', '.png')):
                    img_path = os.path.join(class_dir, img_name)
                    self.samples.append(img_path)
                    self.labels.append(label_idx)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path = self.samples[idx]
        label = self.labels[idx]

        try:
            image = Image.open(img_path).convert('RGB')
        except:
            # 如果图片损坏，返回一个随机图片
            image = Image.new('RGB', (Config.image_size, Config.image_size))

        if self.transform:
            image = self.transform(image)

        return image, label


def get_transforms(mode='train'):
    """获取数据增强和预处理 transforms"""

    if mode == 'train':
        return transforms.Compose([
            transforms.RandomResizedCrop(Config.image_size, scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.2),
            transforms.RandomRotation(30),
            transforms.ColorJitter(brightness=0.2, contrast=0.2,
                                  saturation=0.2, hue=0.1),
            transforms.RandomGrayscale(p=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225]),
            transforms.RandomErasing(p=0.2, scale=(0.02, 0.2))
        ])
    else:
        return transforms.Compose([
            transforms.Resize(int(Config.image_size * 1.15)),
            transforms.CenterCrop(Config.image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])


# ============================================================
# 模型定义
# ============================================================

class FlowerClassifier(nn.Module):
    """基于预训练模型的花卉分类器"""

    def __init__(self, num_classes=5, model_name='resnet50', dropout=0.5):
        super(FlowerClassifier, self).__init__()

        self.model_name = model_name

        # 使用不同的预训练模型
        if model_name == 'resnet50':
            self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
            num_features = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()

        elif model_name == 'efficientnet_b0':
            self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
            num_features = self.backbone.classifier[1].in_features
            self.backbone.classifier = nn.Identity()

        elif model_name == 'vit_b_16':
            self.backbone = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
            num_features = self.backbone.heads.head.in_features
            self.backbone.heads = nn.Identity()

        # 分类头
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(num_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(512, num_classes)
        )

        # 初始化最后一个全连接层
        nn.init.normal_(self.classifier[-1].weight, 0, 0.01)
        nn.init.constant_(self.classifier[-1].bias, 0)

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

    def freeze_backbone(self):
        """冻结骨干网络"""
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self, unfreeze_layers=None):
        """
        解冻骨干网络
        参数：
            unfreeze_layers: 从最后一层开始向前数多少层需要解冻
        """
        if unfreeze_layers is None:
            # 解冻所有层
            for param in self.backbone.parameters():
                param.requires_grad = True
        else:
            # 部分解冻
            if self.model_name == 'resnet50':
                layers = ['layer1', 'layer2', 'layer3', 'layer4']
                for layer in layers[-unfreeze_layers:]:
                    for param in getattr(self.backbone, layer).parameters():
                        param.requires_grad = True


# ============================================================
# 训练与评估
# ============================================================

class Trainer:
    """训练器类"""

    def __init__(self, model, train_loader, val_loader, config):
        self.model = model.to(config.device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.config = config

        # 损失函数
        self.criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

        # 优化器
        self.optimizer = optim.AdamW(
            self.model.parameters(),
            lr=config.learning_rate,
            weight_decay=0.01
        )

        # 学习率调度器
        self.scheduler = optim.lr_scheduler.OneCycleLR(
            self.optimizer,
            max_lr=config.learning_rate * 10,
            epochs=config.num_epochs,
            steps_per_epoch=len(train_loader),
            pct_start=0.1,
            anneal_strategy='cos'
        )

        # 最佳模型
        self.best_acc = 0.0
        self.history = {'train_loss': [], 'train_acc': [],
                       'val_loss': [], 'val_acc': []}

    def train_epoch(self, epoch):
        """训练一个epoch"""
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(self.train_loader, desc=f'Epoch {epoch+1}')
        for images, labels in pbar:
            images = images.to(self.config.device)
            labels = labels.to(self.config.device)

            self.optimizer.zero_grad()

            outputs = self.model(images)
            loss = self.criterion(outputs, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.optimizer.step()
            self.scheduler.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            pbar.set_postfix({
                'loss': f'{running_loss/(pbar.n+1):.4f}',
                'acc': f'{100.*correct/total:.2f}%',
                'lr': f'{self.scheduler.get_last_lr()[0]:.6f}'
            })

        epoch_loss = running_loss / len(self.train_loader)
        epoch_acc = 100. * correct / total

        return epoch_loss, epoch_acc

    def evaluate(self):
        """在验证集上评估"""
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in self.val_loader:
                images = images.to(self.config.device)
                labels = labels.to(self.config.device)

                outputs = self.model(images)
                loss = self.criterion(outputs, labels)

                running_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / len(self.val_loader)
        epoch_acc = 100. * correct / total

        return epoch_loss, epoch_acc

    def train(self):
        """完整训练流程"""
        print(f"使用设备: {self.config.device}")
        print(f"训练轮数: {self.config.num_epochs}")

        for epoch in range(self.config.num_epochs):
            train_loss, train_acc = self.train_epoch(epoch)
            val_loss, val_acc = self.evaluate()

            # 记录历史
            self.history['train_loss'].append(train_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_loss'].append(val_loss)
            self.history['val_acc'].append(val_acc)

            print(f"Epoch {epoch+1}/{self.config.num_epochs}: "
                  f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, "
                  f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

            # 保存最佳模型
            if val_acc > self.best_acc:
                self.best_acc = val_acc
                self.save_checkpoint('best_model.pth', epoch, val_acc)
                print(f"  -> 保存最佳模型，准确率: {val_acc:.2f}%")

            # 定期保存checkpoint
            if (epoch + 1) % 10 == 0:
                self.save_checkpoint(f'checkpoint_epoch_{epoch+1}.pth', epoch, val_acc)

        print(f"\n训练完成！最佳验证准确率: {self.best_acc:.2f}%")
        return self.history

    def save_checkpoint(self, path, epoch, acc):
        """保存模型检查点"""
        torch.save({
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'best_acc': acc,
            'history': self.history
        }, path)

    def load_checkpoint(self, path):
        """加载模型检查点"""
        checkpoint = torch.load(path, map_location=self.config.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        self.best_acc = checkpoint['best_acc']
        self.history = checkpoint['history']
        return checkpoint['epoch']


# ============================================================
# 预测与可视化
# ============================================================

class Predictor:
    """预测器"""

    def __init__(self, model_path, config):
        self.model = FlowerClassifier(num_classes=config.num_classes)
        checkpoint = torch.load(model_path, map_location=config.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model = self.model.to(config.device)
        self.model.eval()
        self.transform = get_transforms(mode='val')
        self.config = config

    def predict(self, image_path):
        """预测单张图片"""
        image = Image.open(image_path).convert('RGB')
        image_tensor = self.transform(image).unsqueeze(0).to(self.config.device)

        with torch.no_grad():
            outputs = self.model(image_tensor)
            probabilities = torch.softmax(outputs, dim=1)[0]
            predicted_class = outputs.argmax(dim=1).item()

        return {
            'class': Config.class_names[predicted_class],
            'confidence': probabilities[predicted_class].item(),
            'all_probabilities': {
                name: prob.item()
                for name, prob in zip(Config.class_names, probabilities)
            }
        }

    def predict_batch(self, image_paths):
        """批量预测"""
        results = []
        for path in image_paths:
            result = self.predict(path)
            result['path'] = path
            results.append(result)
        return results


def plot_training_history(history):
    """绘制训练历史"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # 损失曲线
    ax1.plot(history['train_loss'], label='Train Loss')
    ax1.plot(history['val_loss'], label='Val Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training and Validation Loss')
    ax1.legend()
    ax1.grid(True)

    # 准确率曲线
    ax2.plot(history['train_acc'], label='Train Acc')
    ax2.plot(history['val_acc'], label='Val Acc')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Training and Validation Accuracy')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150)
    plt.show()


# ============================================================
# 主程序
# ============================================================

if __name__ == "__main__":
    # 准备数据
    print("准备数据集...")
    train_dataset = FlowerDataset(Config.train_dir, get_transforms('train'), mode='train')
    val_dataset = FlowerDataset(Config.val_dir, get_transforms('val'), mode='val')

    train_loader = DataLoader(train_dataset, batch_size=Config.batch_size,
                             shuffle=True, num_workers=Config.num_workers,
                             pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.batch_size,
                           shuffle=False, num_workers=Config.num_workers,
                           pin_memory=True)

    print(f"训练集: {len(train_dataset)} 样本")
    print(f"验证集: {len(val_dataset)} 样本")

    # 创建模型
    print("\n创建模型...")
    model = FlowerClassifier(num_classes=Config.num_classes, model_name='resnet50')

    # 迁移学习策略：先冻结 backbone 训练分类头
    print("阶段1: 训练分类头...")
    model.freeze_backbone()
    trainer = Trainer(model, train_loader, val_loader, Config)
    trainer.train()

    # 阶段2：解冻 backbone，微调
    print("\n阶段2: 微调整个网络...")
    model.unfreeze_backbone()
    trainer = Trainer(model, train_loader, val_loader, Config)
    trainer.optimizer = optim.AdamW(
        model.parameters(),
        lr=Config.learning_rate * 0.1,  # 降低学习率
        weight_decay=0.01
    )
    trainer.scheduler = optim.lr_scheduler.CosineAnnealingLR(
        trainer.optimizer, T_max=Config.num_epochs
    )
    trainer.train()

    # 绘制训练曲线
    plot_training_history(trainer.history)

    # 预测示例
    print("\n预测示例:")
    predictor = Predictor('best_model.pth', Config)
    # result = predictor.predict('path/to/flower/image.jpg')
    # print(json.dumps(result, indent=2))


### 超参数搜索实战


In [ ]:
import optuna
import torch
import torch.nn as nn
from sklearn.model_selection import cross_val_score
import numpy as np


def objective(trial):
    """
    使用Optuna进行超参数搜索
    这是一个简化版本，实际使用时需要根据具体任务调整
    """

    # 定义超参数搜索空间
    lr = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
    dropout = trial.suggest_float('dropout', 0.1, 0.7)
    num_layers = trial.suggest_int('num_layers', 2, 6)
    hidden_size = trial.suggest_categorical('hidden_size', [128, 256, 512, 1024])

    # 创建模型
    model = build_model(
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout
    )

    # 创建数据加载器
    train_loader, val_loader = get_data_loaders(batch_size)

    # 训练
    trainer = Trainer(model, train_loader, val_loader, lr)
    history = trainer.train(epochs=10, verbose=False)

    # 返回验证集上的最佳准确率
    return max(history['val_acc'])


def main():
    # 创建研究
    study = optuna.create_study(
        direction='maximize',
        study_name='deep_learning_hyperparameter_search',
        storage='sqlite:///study.db',
        load_if_exists=True
    )

    # 运行优化
    study.optimize(objective, n_trials=50, show_progress_bar=True)

    # 输出最佳结果
    print(f"\n最佳试验:")
    print(f"  准确率: {study.best_value:.4f}")
    print(f"  参数: {study.best_params}")

    # 可视化
    optuna.visualization.plot_optimization_history(study)
    optuna.visualization.plot_parallel_coordinate(study)
    optuna.visualization.plot_param_importances(study)


if __name__ == "__main__":
    main()


## 练习题

1. **项目设计**：假设你需要构建一个垃圾分类系统，垃圾图片来自用户手机拍摄，环境复杂多变。请列出完整的数据收集和预处理方案，包括如何处理光照变化、角度差异、遮挡等问题。

2. **迁移学习策略**：如果要将在ImageNet上预训练的模型迁移到医学影像分类任务（如X光片识别），应该采用什么样的微调策略？为什么不能直接用预训练模型的输出层？

3. **数据不平衡处理**：假设你正在训练一个疾病检测模型，但正样本（患病）只占2%，负样本占98%。请列出至少5种处理数据不平衡的策略，并比较它们的优缺点。

4. **模型选择**：某公司需要在边缘设备（手机）上部署图像分类模型，延迟要求<50ms，精度要求>85%。请设计一个模型选择和压缩方案，包括模型架构选择、量化、剪枝等技术的应用。

5. **错误分析**：模型在测试集上的准确率为85%，但你发现它在某些特定类别上表现很差（如将"猫"误判为"狗"的错误占了总错误的30%）。请设计一套错误分析流程，并提出针对性的改进方案。

## 延伸阅读

- He, K., et al. (2019). "Bag of Tricks for Image Classification with Convolutional Neural Networks". CVPR 2019.
- Howard, J., & Gugger, S. (2020). "Deep Learning for Coders with Fastai and PyTorch". O'Reilly.
- Liu, G., et al. (2022). "A Survey of Deep Learning for Image Inpainting". IJCV.
- Polyak, A., & Taylor, L. (2021). "Training Data Validation and Sanitization for Deep Learning". arXiv.
- Krizhevsky, A. (2009). "Learning Multiple Layers of Features from Tiny Images". Technical Report.

---

## 附录：深度学习术语表

| 术语 | 中文 | 说明 |
|-----|------|------|
| Epoch | 轮次 | 整个训练数据集遍历一次 |
| Batch | 批次 | 一次迭代使用的样本数 |
| Iteration | 迭代 | 参数更新一次 |
| Learning Rate | 学习率 | 参数更新的步长 |
| Momentum | 动量 | 加速收敛的优化技术 |
| Weight Decay | 权重衰减 | L2正则化 |
| Dropout | Dropout | 随机丢弃神经元防止过拟合 |
| Batch Normalization | 批归一化 | 规范化批次数据 |
| Gradient Descent | 梯度下降 | 最优化算法 |
| Backpropagation | 反向传播 | 计算梯度的算法 |

---

**文档版本**：v1.0
**创建日期**：2026年6月8日
**适用对象**：深度学习初学者，具有Python编程基础和线性代数基础
**前置知识**：Week 1-4内容（Python编程、机器学习基础、数学基础）
